# 04 — Joins & Unions

Join types, unions, deduplication, and aliasing. Uses synthetic data since `vendas` is a single table.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Synthetic data

Two small tables to join.

In [ ]:
import pandas as pd

vendas = session.createDataFrame(pd.DataFrame({
    "id": [1, 2, 3, 4],
    "cidade": ["SP", "RJ", "MG", "SP"],
    "valor": [100.0, 200.0, 150.0, 300.0],
}))

clientes = session.createDataFrame(pd.DataFrame({
    "id": [1, 2, 3, 5],
    "nome": ["Ana", "Bruno", "Carlos", "Diana"],
    "estado": ["SP", "RJ", "MG", "BA"],
}))

vendas.show()
clientes.show()

## 2. Inner join

In [ ]:
vendas.join(clientes, "id", "inner").show()

## 3. Left / Right / Full outer

In [ ]:
vendas.join(clientes, "id", "left").show()
vendas.join(clientes, "id", "right").show()
vendas.join(clientes, "id", "full").show()

## 4. Left semi / Left anti

In [ ]:
vendas.join(clientes, "id", "left_semi").show()
vendas.join(clientes, "id", "left_anti").show()

## 5. Cross join

In [ ]:
vendas.crossJoin(clientes).show()

## 6. Union / UnionAll / UnionByName

In [ ]:
a = session.createDataFrame(pd.DataFrame({"x": [1, 2], "y": [10, 20]}))
b = session.createDataFrame(pd.DataFrame({"x": [3, 4], "y": [30, 40]}))

a.union(b).show()
a.unionAll(b).show()

# unionByName reorders by column name
c = session.createDataFrame(pd.DataFrame({"y": [50, 60], "x": [5, 6]}))
a.unionByName(c).show()

## 7. Distinct & dropDuplicates

In [ ]:
dup = session.createDataFrame(pd.DataFrame({"a": [1, 1, 2, 2, 3], "b": [10, 10, 20, 20, 30]}))
dup.distinct().show()
dup.dropDuplicates().show()
dup.dropDuplicates(["a"]).show()

## 8. Alias

In [ ]:
vendas.alias("v").join(clientes.alias("c"), "id").select("v.cidade", "c.nome").show()

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")